In [ ]:
import numpy as np
import pandas as pd
import ibis
from utils.f_0_dirs import get_data_dirs

# Generate a W interactions matrix with numpy
# \begin{align}
#   w_{ij}\in \mathbf{W}_g &=
#     \begin{cases}
#       0,             & \text{if } i=j \\
#       0,             & \text{if } w_{ij,k} \neq 0, \exists k<g \\
#       \frac{1}{n_g-1}, & \text{else if } (i,j)\in g \\
#       0,             & \text{otherwise}
#     \end{cases}
# \end{align}

fixed_name = "working_fixed"
dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)
t_fixed = (
    con.table(fixed_name)
    .select('registered_number', 'pc8', 'pc4', 'ttwa')
)

# Generate an n \times n matrix
# With 1s in the off-diagonal elements for each group g, and 0s elsewhere
# G
def generate_W_matrix(df: pd.DataFrame, group_col: str) -> np.ndarray:
    n = len(df)
    W = np.zeros((n, n))
    for group in df[group_col].unique():
        indices = df.index[df[group_col] == group].tolist()
        for i in indices:
            for j in indices:
                if i != j:
                    W[i, j] = 1 / (len(indices) - 1)
    return W

df_fixed = t_fixed.execute()
print(f"Generating interaction matrix W for {len(df_fixed)} observations grouped by pc8...")
W_pc8 = generate_W_matrix(df_fixed, 'pc8')
print(W_pc8)

Generating interaction matrix W for 152379 observations grouped by pc8...


MemoryError: Unable to allocate 173. GiB for an array with shape (152379, 152379) and data type float64